# 05 — GRU sequence model vs tabular baselines (spec §3b/§5)

Fair-comparison rules: same ≥2-purchase population as 03b, same frozen temporal split, same `ds_common` metric suite, baselines re-scored on the **same test rows** (from `baseline_test_scores.pkl`, produced by 04). Sequence length L selected on **val** from {10, 20}; test scored once.

**Backend note:** the GRU is Keras 3 on the **PyTorch backend** — TensorFlow wheels (2.20/2.21) segfault on import in this env (anaconda py3.13/arm64), so the framework choice is an environment constraint, honestly recorded. Padded timesteps are all-zero vectors handled by a `Masking` layer (real timesteps are standardized and never all-zero).

**Dependency:** run 03b (sequences.npz) and 04 (baseline_* artifacts) first. **Determinism note:** seeds set via `keras.utils.set_random_seed`, but CPU RNN training is only approximately reproducible run-to-run; report numbers from the saved run.

In [1]:
# Bootstrap: Keras 3 on the torch backend (must set env var BEFORE importing keras)
import os; os.environ.setdefault("KERAS_BACKEND","torch")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK","1")   # M1: fall back to CPU for ops MPS lacks (e.g. linalg_qr)
import importlib.util, sys, subprocess
for pkg in ["torch","keras"]:
    if importlib.util.find_spec(pkg) is None:
        print("installing",pkg,"..."); subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
import keras, torch
assert keras.backend.backend()=="torch", "KERAS_BACKEND must be torch (restart kernel if this fails)"
print("keras",keras.__version__,"| torch",torch.__version__,"| backend",keras.backend.backend())

keras 3.15.0 | torch 2.8.0 | backend torch


In [2]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from ds_common import *
keras.utils.set_random_seed(RANDOM_SEED)
seq=load_sequences()
BTS=PROC_DIR/"baseline_test_scores.pkl"
assert BTS.exists(), "run 04_baseline first (it saves baseline_test_scores.pkl)"
bts=pd.read_pickle(BTS)
Xs,Ms,St=seq["X_seq"],seq["X_mask"],seq["X_static"].copy()
y=seq["y"].astype("float32"); sp=seq["split"]
tr,va,te=(sp=="train"),(sp=="val"),(sp=="test")
print("splits:",{k:int(v) for k,v in zip(*np.unique(sp,return_counts=True))},"| baseline test rows:",len(bts))

loaded sequences.npz: n=35,713 L=20 F=7 | splits {np.str_('test'): np.int64(11627), np.str_('train'): np.int64(16384), np.str_('val'): np.int64(7702)}
splits: {np.str_('test'): 11627, np.str_('train'): 16384, np.str_('val'): 7702} | baseline test rows: 13961


## 1. Preprocessing — fit on train only
Static: impute NaN (train median; `has_return_history` is the missing flag) then standardize. Sequence channels: standardize using **masked train positions only**, then restore padding to exact 0 (spec §7) — padded steps become all-zero vectors, which the `Masking` layer skips.

In [3]:
med=np.nanmedian(St[tr],axis=0); ix=np.where(np.isnan(St)); St[ix]=np.take(med,ix[1])
mu,sd=St[tr].mean(0),St[tr].std(0)+1e-6; St=((St-mu)/sd).astype("float32")
m3=Ms[:,:,None]
mu_s=(Xs[tr]*m3[tr]).sum((0,1))/m3[tr].sum((0,1))
sd_s=np.sqrt(((Xs[tr]-mu_s)**2*m3[tr]).sum((0,1))/m3[tr].sum((0,1)))+1e-6
Xn=(((Xs-mu_s)/sd_s)*m3).astype("float32")
print("seq channels standardized on masked train positions; padding restored to 0 (all-zero => masked)")

seq channels standardized on masked train positions; padding restored to 0 (all-zero => masked)


## 2. Model builder + L selection on val ({10, 20})

In [4]:
def build(L,F,S):
    si=keras.Input((L,F)); sti=keras.Input((S,))
    x=keras.layers.Masking(mask_value=0.0)(si)
    h=keras.layers.GRU(48, recurrent_initializer="glorot_uniform")(x)   # avoid linalg_qr (orthogonal init) on MPS
    s=keras.layers.Dense(16,activation="relu")(sti)
    z=keras.layers.Concatenate()([h,s]); z=keras.layers.Dropout(0.2)(z)
    out=keras.layers.Dense(1,activation="sigmoid")(keras.layers.Dense(32,activation="relu")(z))
    m=keras.Model([si,sti],out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3),loss="binary_crossentropy",
              metrics=[keras.metrics.AUC(curve="PR",name="aucpr")])
    return m
def fit_eval(L):
    Xl=Xn[:,-L:,:]
    keras.backend.clear_session(); keras.utils.set_random_seed(RANDOM_SEED)
    m=build(L,Xl.shape[2],St.shape[1])
    es=keras.callbacks.EarlyStopping(monitor="val_aucpr",mode="max",patience=5,restore_best_weights=True)
    m.fit([Xl[tr],St[tr]],y[tr],validation_data=([Xl[va],St[va]],y[va]),
          epochs=40,batch_size=256,verbose=0,callbacks=[es])
    pv=m.predict([Xl[va],St[va]],batch_size=2048,verbose=0).ravel()
    from sklearn.metrics import average_precision_score, roc_auc_score
    return m,average_precision_score(y[va],pv),roc_auc_score(y[va],pv)
results={}
for L in [10,20]:
    m,vpr,vroc=fit_eval(L); results[L]=(m,vpr,vroc)
    print(f"L={L}: val PR-AUC {vpr:.4f} | val ROC {vroc:.4f}")
BEST_L=max(results,key=lambda L:results[L][1]); gru=results[BEST_L][0]
print("selected L =",BEST_L,"(on val PR-AUC)")

L=10: val PR-AUC 0.7402 | val ROC 0.7643
L=20: val PR-AUC 0.7408 | val ROC 0.7663
selected L = 20 (on val PR-AUC)


## 3. Test — scored once; baselines on the SAME rows

In [5]:
Xl=Xn[:,-BEST_L:,:]
p_gru=gru.predict([Xl[te],St[te]],batch_size=2048,verbose=0).ravel()
yte=y[te].astype(int)
key=pd.DataFrame({KEY:seq["customer_id"][te],"cutoff":pd.to_datetime(seq["cutoff"][te])})
al=key.merge(bts,on=[KEY,"cutoff"],how="left",validate="one_to_one")
assert al["p_logreg"].notna().all() and (al["churn"].values==yte).all(), "baseline alignment failed"
rows=[evaluate_classifier("gru",yte,p_gru)]
for nm,col in [("logreg","p_logreg"),("boosted","p_boosted"),("rf_pruned","p_rf_pruned")]:
    rows.append(evaluate_classifier(nm+"_same_subset",yte,al[col].values))
cmp=pd.DataFrame(rows)
print(cmp[["model","ROC_AUC","PR_AUC","PR_AUC_base","P@10%","Lift@10%","Brier","LogLoss","ECE"]].to_string(index=False))

                model  ROC_AUC  PR_AUC  PR_AUC_base  P@10%  Lift@10%  Brier  LogLoss    ECE
                  gru   0.7581  0.6617       0.4176  0.750      1.80 0.2099   0.6213 0.1125
   logreg_same_subset   0.7526  0.6500       0.4176  0.734      1.76 0.2108   0.6078 0.1078
  boosted_same_subset   0.7406  0.6344       0.4176  0.731      1.75 0.2192   0.6312 0.1215
rf_pruned_same_subset   0.7468  0.6411       0.4176  0.742      1.78 0.2050   0.5897 0.0670


## 4. Is the GRU difference real? Customer-clustered bootstrap of ΔROC / ΔPR (GRU − LogReg)

In [6]:
rng=np.random.default_rng(RANDOM_SEED)
from sklearn.metrics import roc_auc_score, average_precision_score
df=pd.DataFrame({"k":seq["customer_id"][te],"y":yte,"a":p_gru,"b":al["p_logreg"].values})
groups={g:s[["y","a","b"]].values for g,s in df.groupby("k")}
gk=np.array(list(groups)); dro=[]; dpr=[]
for _ in range(500):
    m=np.concatenate([groups[g] for g in rng.choice(gk,size=len(gk),replace=True)])
    if m[:,0].min()==m[:,0].max(): continue
    dro.append(roc_auc_score(m[:,0],m[:,1])-roc_auc_score(m[:,0],m[:,2]))
    dpr.append(average_precision_score(m[:,0],m[:,1])-average_precision_score(m[:,0],m[:,2]))
print("dROC(gru-logreg) point %.4f CI95 (%.4f, %.4f)"%(roc_auc_score(yte,p_gru)-roc_auc_score(yte,al['p_logreg'].values),*np.percentile(dro,[2.5,97.5])))
print("dPR (gru-logreg) point %.4f CI95 (%.4f, %.4f)"%(average_precision_score(yte,p_gru)-average_precision_score(yte,al['p_logreg'].values),*np.percentile(dpr,[2.5,97.5])))
print("Reading: CI excluding 0 = a real difference on this population; otherwise report a statistical tie honestly (spec: DL must beat the baseline honestly or say so).")

dROC(gru-logreg) point 0.0055 CI95 (0.0004, 0.0101)
dPR (gru-logreg) point 0.0117 CI95 (0.0022, 0.0203)
Reading: CI excluding 0 = a real difference on this population; otherwise report a statistical tie honestly (spec: DL must beat the baseline honestly or say so).


## 5. Persist GRU artifacts

In [7]:
gru.save(PROC_DIR/"gru.keras")
gs=pd.DataFrame({KEY:seq["customer_id"][te],"cutoff":pd.to_datetime(seq["cutoff"][te]),"churn":yte,"p_gru":p_gru})
gs.to_pickle(PROC_DIR/"gru_test_scores.pkl")
cmp.to_pickle(PROC_DIR/"gru_comparison_table.pkl")
np.savez(PROC_DIR/"gru_preproc.npz",static_median=med,static_mu=mu,static_sd=sd,seq_mu=mu_s,seq_sd=sd_s,best_L=BEST_L)
print("saved -> gru.keras, gru_test_scores.pkl, gru_comparison_table.pkl, gru_preproc.npz")

saved -> gru.keras, gru_test_scores.pkl, gru_comparison_table.pkl, gru_preproc.npz


## 6. Capacity ablation — is the ceiling model capacity or signal? (VAL-ONLY; test untouched)

If capacity were the bottleneck, bigger/deeper recurrent models would beat GRU(48) on val. Four variants under the **identical protocol** (same L, same early stopping, same seed): GRU(48) reference, GRU(128), stacked 2×GRU(48), GRU(48)+2-head self-attention. **Selection/report on val only** — the single test evaluation above remains the only test look.

In [8]:
gi=dict(recurrent_initializer="glorot_uniform")
def build_variant(kind,L,F,S):
    si=keras.Input((L,F)); sti=keras.Input((S,))
    x=keras.layers.Masking(mask_value=0.0)(si)
    if kind=="gru48": h=keras.layers.GRU(48,**gi)(x)
    elif kind=="gru128": h=keras.layers.GRU(128,**gi)(x)
    elif kind=="stacked48x2":
        h=keras.layers.GRU(48,**gi)(keras.layers.GRU(48,return_sequences=True,**gi)(x))
    elif kind=="gru48_attn":
        seqh=keras.layers.GRU(48,return_sequences=True,**gi)(x)
        att=keras.layers.MultiHeadAttention(num_heads=2,key_dim=24)(seqh,seqh)
        h=keras.layers.GlobalAveragePooling1D()(att)
    s=keras.layers.Dense(16,activation="relu")(sti)
    zz=keras.layers.Concatenate()([h,s]); zz=keras.layers.Dropout(0.2)(zz)
    out=keras.layers.Dense(1,activation="sigmoid")(keras.layers.Dense(32,activation="relu")(zz))
    m=keras.Model([si,sti],out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3),loss="binary_crossentropy",
              metrics=[keras.metrics.AUC(curve="PR",name="aucpr")])
    return m
from sklearn.metrics import roc_auc_score, average_precision_score
Xl=Xn[:,-BEST_L:,:]; rows_c=[]
for kind in ["gru48","gru128","stacked48x2","gru48_attn"]:
    keras.backend.clear_session(); keras.utils.set_random_seed(RANDOM_SEED)
    m=build_variant(kind,BEST_L,Xl.shape[2],St.shape[1])
    es=keras.callbacks.EarlyStopping(monitor="val_aucpr",mode="max",patience=5,restore_best_weights=True)
    m.fit([Xl[tr],St[tr]],y[tr],validation_data=([Xl[va],St[va]],y[va]),
          epochs=40,batch_size=256,verbose=0,callbacks=[es])
    pv=m.predict([Xl[va],St[va]],batch_size=2048,verbose=0).ravel()
    rows_c.append(dict(variant=kind,params=m.count_params(),
        val_ROC=round(roc_auc_score(y[va],pv),4),val_PR=round(average_precision_score(y[va],pv),4)))
ab=pd.DataFrame(rows_c); ref=ab.loc[ab["variant"]=="gru48","val_PR"].iat[0]
ab["dPR_vs_gru48"]=(ab["val_PR"]-ref).round(4)
print(ab.to_string(index=False))
ab.to_pickle(PROC_DIR/"gru_capacity_ablation.pkl")
print("\nReading: if bigger/deeper variants sit within ~±0.005 val PR of GRU(48), capacity is NOT the bottleneck —")
print("the ~0.76 line is a feature/signal ceiling (median sequence length is 4-8 events; no marketing/browsing covariates in the data).")

    variant  params  val_ROC  val_PR  dPR_vs_gru48
      gru48   10433   0.7663  0.7408        0.0000
     gru128   57393   0.7628  0.7411        0.0003
stacked48x2   24545   0.7636  0.7396       -0.0012
 gru48_attn   19841   0.7691  0.7386       -0.0022

Reading: if bigger/deeper variants sit within ~±0.005 val PR of GRU(48), capacity is NOT the bottleneck —
the ~0.76 line is a feature/signal ceiling (median sequence length is 4-8 events; no marketing/browsing covariates in the data).


## Takeaways (filled from this run's printed tables)

- **The GRU wins on the same ≥2-purchase test rows** (test scored once): ROC 0.7581 / PR 0.6617 / P@10% 0.750 / Lift@10% 1.80 vs logreg 0.7526 / 0.6500 / 0.734 / 1.76 (boosted 0.7406, pruned RF 0.7468). Customer-clustered bootstrap: ΔROC CI95 (0.0004, 0.0101), ΔPR CI95 (0.0022, 0.0203) — **both exclude 0**. The sequence signal is small but statistically real, even at a median of 4–8 events per customer.

- **The ceiling is signal, not capacity** (§6, val-only): 5.5× parameters (+0.0003), extra depth (−0.0012) and self-attention (−0.0022) all sit within ±0.005 val PR of GRU(48). Raising ~0.76 needs new information sources (marketing touches, browsing, demographics), not architecture.

- **Model positioning (final):** the GRU delivers a small but statistically detectable **ranking** lift among customers with ≥2 purchase history (ΔROC/ΔPR CIs exclude 0). It does **not** beat the tabular baselines on probability quality (GRU ECE 0.1125 / Brier 0.2099 vs pruned RF 0.0670 / 0.2050 on the same subset), and it cannot cover single-purchase customers. The GRU therefore serves as **sequence-signal evidence**; the recommended deployment remains the **full-population tabular model** (logreg — best full-test ROC 0.7637 and complete at-risk coverage; pruned RF as the best-calibrated option).